In [0]:
from pyspark.sql import functions as F

# Read the  taxi table and print basic info: row count, column count, schema, and summary stats
df = spark.read.table("group3_gp.testing.green")
print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()
df.describe().show()

In [0]:
%sql
    
-- ════════════════════════════════════════════════
-- Green — Bad data checks
-- Counts rows failing each quality rule in one pass
-- ════════════════════════════════════════════════

-- Row count
SELECT 'Total rows' AS check_name, COUNT(*) AS result
FROM group3_gp.testing.green

UNION ALL

-- Bad fares
-- EXCEPTION: payment_type 3 = No charge, 6 = Voided — zero fare is CORRECT
SELECT 'Bad fares (excl. no-charge & voided)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE CAST(fare_amount AS DOUBLE) <= 0
  AND payment_type NOT IN ('3', '6')

UNION ALL

-- Legitimate zero fares
SELECT 'Legitimate zero fares (no-charge/voided)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE CAST(fare_amount AS DOUBLE) <= 0
  AND payment_type IN ('3', '6')

UNION ALL

-- Bad distances
SELECT 'Bad distances (zero or negative)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE CAST(trip_distance AS DOUBLE) <= 0

UNION ALL

-- Bad timestamps — dropoff before or equal to pickup
SELECT 'Bad timestamps (dropoff <= pickup)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE TO_TIMESTAMP(lpep_dropoff_datetime) <= TO_TIMESTAMP(lpep_pickup_datetime)
UNION ALL

-- Invalid RatecodeID — 99 = Null/unknown per data dictionary
SELECT 'Invalid ratecode (99 = unknown)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE TRY_CAST(ratecodeid AS INT) = 99

UNION ALL

-- Invalid passenger count — TRY_CAST to DOUBLE to handle "2.0" style strings
SELECT 'Invalid passenger count (< 1 or > 8)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.green
WHERE TRY_CAST(passenger_count AS INT) NOT BETWEEN 1 AND 8

In [0]:
%sql
-- Green — Trip type breakdown
-- Maps raw trip_type codes to labels (1 = Street hail, 2 = Dispatch)
SELECT 'Total rows' AS trip_type, NULL AS trip_type_desc, COUNT(*) AS trip_count, NULL AS pct
FROM group3_gp.testing.green

UNION ALL 

SELECT
    trip_type,
    CASE trip_type
        WHEN '1' THEN 'Street hail'
        WHEN '2' THEN 'Dispatch'
        WHEN '1.0' THEN 'Street hail'
        WHEN '2.0' THEN 'Dispatch'
        ELSE 'Unknown'
    END AS trip_type_desc,
    COUNT(*) AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM group3_gp.testing.green
GROUP BY trip_type
ORDER BY trip_count DESC

In [0]:
%sql
-- Green — Year distribution
-- Shows trip volume per year to spot coverage gaps
SELECT
    YEAR(TO_TIMESTAMP(lpep_pickup_datetime)) AS pickup_year,
    COUNT(*) AS trip_count
FROM group3_gp.testing.green
GROUP BY pickup_year
ORDER BY pickup_year

In [0]:
%sql
    
-- Green — Payment type breakdown
SELECT
    payment_type,
    CASE payment_type
        WHEN '0' THEN 'Flex fare'
        WHEN '1' THEN 'Credit card'
        WHEN '2' THEN 'Cash'
        WHEN '3' THEN 'No charge'
        WHEN '4' THEN 'Dispute'
        WHEN '5' THEN 'Unknown'
        WHEN '6' THEN 'Voided trip'
        WHEN '1.0' THEN 'Credit card'
        WHEN '2.0' THEN 'Cash'
        WHEN '3.0' THEN 'No charge'
        WHEN '4.0' THEN 'Dispute'
        WHEN '5.0' THEN 'Unknown'
        ELSE 'Unrecognised'
    END AS payment_description,
    COUNT(*) AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM group3_gp.testing.green
GROUP BY payment_type
ORDER BY payment_type DESC


In [0]:
%sql
    
SELECT
  VendorID,
  CASE VendorID
    WHEN '1' THEN 'Creative Mobile Technologies'
    WHEN '2' THEN 'Curb Mobility'
    WHEN '6' THEN 'Myle Technologies'
    ELSE 'Unknown'
  END AS VendorName,
  COUNT(*) AS Vendor_count  
FROM group3_gp.testing.green
GROUP BY VendorID
ORDER BY VendorID ASC
